# 01 — Scraper Experiments

Iterate on the scraper: try different content selectors, text cleaners,
and inspect output quality before committing to a full crawl.

**Goal:** find the selector strategy that yields the cleanest, most complete text per page.

In [ ]:
import sys, os
from pathlib import Path

# app/ is now at repo_root/RAG_Ai_Assistant_Uni/app/
repo_root = str(Path(os.getcwd()).parents[1] / "RAG_Ai_Assistant_Uni")
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse

BASE_URL = "https://alphawave.hr/"

## 1. Inspect a Single Page

Raw output from the current scraper on the homepage.

In [ ]:
from app.scraper import scrape_page

result = scrape_page(BASE_URL)
print(f"Title : {result['title']}")
print(f"Length: {len(result['content'])} chars")
print(f"Lines : {result['content'].count(chr(10))}")
print("\n--- First 1000 chars ---")
print(result['content'][:1000])

## 2. Alternative Content Selectors

Compare `<main>`, `<article>`, `<body>`, and a custom selector.
Pick the one that gives the most signal with the least noise.

In [ ]:
STRIP_TAGS = ["script", "style", "noscript", "nav", "header", "footer"]
SELECTORS = ["main", "article", "section", None]  # None = full body

response = requests.get(BASE_URL)
response.raise_for_status()

def extract_with_selector(html: str, selector: str | None) -> str:
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(STRIP_TAGS):
        tag.decompose()
    root = soup.find(selector) if selector else soup
    if root is None:
        return ""
    lines = [l.strip() for l in root.get_text(separator="\n").splitlines()]
    return "\n".join(l for l in lines if l)

for sel in SELECTORS:
    text = extract_with_selector(response.text, sel)
    label = sel or "body (full)"
    print(f"[{label:12}] {len(text):6} chars  |  first 120: {text[:120]!r}")
    print()

## 3. Strip-Tag Sensitivity

Check what content is removed by each stripped tag.

In [ ]:
soup_full = BeautifulSoup(response.text, "html.parser")
full_text_len = len(soup_full.get_text())

for tag in STRIP_TAGS:
    soup = BeautifulSoup(response.text, "html.parser")
    for t in soup([tag]):
        t.decompose()
    reduced = len(soup.get_text())
    removed = full_text_len - reduced
    print(f"Removing <{tag:8}>: -{removed:5} chars ({removed/full_text_len*100:.1f}%)")

## 4. Dry-Run Crawl (no DB writes)

Crawl the whole site, collect URL list and content stats without inserting into the DB.
Set `MAX_PAGES` to limit depth during exploration.

In [ ]:
from app.scraper import scrape_page, extract_internal_links

MAX_PAGES = 20  # increase for a full crawl

visited, to_visit = set(), {BASE_URL}
pages = []

while to_visit and len(visited) < MAX_PAGES:
    url = to_visit.pop()
    if url in visited:
        continue
    visited.add(url)
    try:
        data = scrape_page(url)
        pages.append({"url": url, "title": data["title"], "length": len(data["content"])})
        resp = requests.get(url)
        soup = BeautifulSoup(resp.text, "html.parser")
        for link in extract_internal_links(soup, BASE_URL):
            if link not in visited:
                to_visit.add(link)
        print(f"OK  {len(data['content']):6} chars  {url}")
    except Exception as e:
        print(f"ERR {url}: {e}")

print(f"\nScraped: {len(pages)} pages")
print(f"Total content : {sum(p['length'] for p in pages):,} chars")
print(f"Avg per page  : {sum(p['length'] for p in pages)//len(pages):,} chars")

## 5. Full Ingest with Custom Chunk Config

Run the full crawl + chunk + insert pipeline with the chosen settings.
Adjust `CHUNK_SIZE` and `OVERLAP` before running.

In [ ]:
CHUNK_SIZE = 800
OVERLAP    = 120

# Uncomment to run — this writes to the DB
# from app.chunking import chunk_text
# from app.database import insert_document, get_connection
#
# visited, to_visit = set(), {BASE_URL}
# total_chunks = 0
#
# while to_visit:
#     url = to_visit.pop()
#     if url in visited:
#         continue
#     visited.add(url)
#     try:
#         data = scrape_page(url)
#         resp = requests.get(url)
#         soup = BeautifulSoup(resp.text, "html.parser")
#         for link in extract_internal_links(soup, BASE_URL):
#             if link not in visited:
#                 to_visit.add(link)
#         chunks = chunk_text(data['content'], chunk_size=CHUNK_SIZE, overlap=OVERLAP)
#         for i, chunk in enumerate(chunks):
#             insert_document(url=url, title=f"{data['title']} (chunk {i+1})", content=chunk)
#         total_chunks += len(chunks)
#         print(f"{len(chunks):3} chunks  {url}")
#     except Exception as e:
#         print(f"ERR {url}: {e}")
#
# print(f"\nDone. Total chunks inserted: {total_chunks}")
print(f"Ready to ingest with chunk_size={CHUNK_SIZE}, overlap={OVERLAP}")